# Analysis, figures and Grad-CAM

CPU only. Reads the outputs of the training runs, which may come from several sessions: attach each session's output as an input and the first cell links all result files into one folder. Grad-CAM also needs the public PlantVillage dataset for its segmented leaf images.

Outputs: `analysis/` (tables, paired tests, calibration), `gradcam/`, `extra/` (per-class metrics, confusion matrices) and `figures/` under `OUT_DIR`.

In [ ]:
import glob, os, shutil
SRC = "../../src"  # on Kaggle: attach this repository and point SRC to its src/ folder
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for f in glob.glob(f"{SRC}/*.py"):
    shutil.copy(f, WORK)
%cd {WORK}

Collect the result, logit and checkpoint files of all runs into `/kaggle/working/revision`.

In [ ]:
import os, glob, json
os.environ['OUT_DIR'] = '/kaggle/working/revision'
dst = '/kaggle/working/revision'
for sub in ('results', 'logits', 'checkpoints'):
    os.makedirs(f'{dst}/{sub}', exist_ok=True)
seen = {}
def result_files(top='/kaggle/input'):
    # skip the image folders: a recursive glob over the PlantVillage images takes minutes
    for root, dirs, files in os.walk(top):
        if root.endswith('/revision/results'):
            yield from (os.path.join(root, f) for f in files if f.endswith('.json'))
            dirs[:] = []
        elif os.path.basename(root) in ('color', 'grayscale', 'segmented') or any(d.startswith(('Apple___', 'Tomato___')) for d in dirs):
            dirs[:] = []
for f in sorted(result_files()):
    name = os.path.basename(f)[:-5]
    r = json.load(open(f))
    if r.get('smoke'):
        continue
    base = os.path.dirname(os.path.dirname(f))
    if name in seen:
        print('DUPLICATE (kept first):', name, '|', seen[name], '|', base)
        continue
    seen[name] = base
    for sub, ext in (('results', '.json'), ('logits', '.npz'), ('checkpoints', '.pth')):
        src, link = f'{base}/{sub}/{name}{ext}', f'{dst}/{sub}/{name}{ext}'
        if not os.path.exists(src):
            print('MISSING', src)
        elif not os.path.exists(link):
            os.symlink(src, link)
from collections import Counter
cfg = Counter(n.rsplit('_seed', 1)[0] for n in seen)
print(len(seen), 'runs found (expected 55)')
for k in sorted(cfg): print(f'  {k}: {cfg[k]} seeds')

Summary tables, paired t-tests with Holm correction, compute cost and calibration.

In [ ]:
!OUT_DIR=/kaggle/working/revision python analysis.py

In [ ]:
import pandas as pd, json, glob
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
A = '/kaggle/working/revision/analysis'
print(pd.read_csv(f'{A}/summary.csv').to_string(index=False))
pc = pd.read_csv(f'{A}/per_condition.csv'); print(pc.to_string(index=False))
# worst condition per configuration (mean over seeds)
rows = []
for f in glob.glob('/kaggle/working/revision/results/*.json'):
    r = json.load(open(f)); cfgname = f"{r['arch']}_{r['variant']}" + ('_os' if r['oversample'] else '')
    rows.append({'config': cfgname, **{k: v['acc'] for k, v in r['results'].items()},
                 'unique_imgs': r.get('mean_unique_train_images'), 'epochs': r['epochs_trained']})
df = pd.DataFrame(rows).groupby('config').mean(numeric_only=True)
conds = [c for c in df.columns if c not in ('clean', 'unique_imgs', 'epochs')]
df['worst_condition'] = df[conds].idxmin(axis=1)
print(df[['clean', 'worst_condition', 'unique_imgs', 'epochs']].round(2).to_string())
df.round(3).to_csv(f'{A}/per_config_conditions.csv')

In [ ]:
t = pd.read_csv(f'{A}/paired_tests.csv')
print(t[['reference','compared','metric','n_seeds','mean_diff_pp','ci95_low','ci95_high','p','p_holm','sig']].round(4).to_string(index=False))
print(pd.read_csv(f'{A}/compute.csv').to_string(index=False))
c = pd.read_csv(f'{A}/calibration.csv'); print(c.to_string(index=False))

Grad-CAM for ResNet50 A0 and A4 on 50 correct and 50 wrong test predictions, with the share of the map that falls on the leaf.

In [ ]:
import os
os.environ['OUT_DIR'] = '/kaggle/working/revision'
# find the PlantVillage "segmented" folder (black background) without walking every image
seg = None
for root, dirs, _ in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'segmented' and dirs:
        seg = root; break
    if root.count(os.sep) >= 7 or any(d.startswith(('Apple___', 'Tomato___')) for d in dirs):
        dirs[:] = []  # class folders reached: nothing deeper can be the segmented root
print('segmented root:', seg)
seg_arg = f'--segmented_root "{seg}"' if seg else ''
for run in ['resnet50_A0_seed0', 'resnet50_A4_seed0']:
    !OUT_DIR=/kaggle/working/revision python gradcam.py --run {run} --n_correct 50 --n_wrong 50 {seg_arg}

Per-class metrics, confusion matrices and reliability bins.

In [ ]:
!OUT_DIR=/kaggle/working/revision python extra_stats.py

Figures 5 to 18 of the paper.

In [ ]:
!OUT_DIR=/kaggle/working/revision python figs.py

In [ ]:
# pack the outputs (without checkpoints)
!cd /kaggle/working/revision && zip -qr /kaggle/working/revision_analysis.zip analysis gradcam results extra figures && ls -la /kaggle/working/revision_analysis.zip
!rm -rf /kaggle/working/revision/checkpoints /kaggle/working/revision/logits  # symlinks only; keeps the output small